# nvidia_cutlass — smoke test

PyPI `nvidia-cutlass` installs `cutlass_cppgen`, `cutlass_library` and `pycute`; `import cutlass` is
the separate CuTe DSL (`nvidia-cutlass-dsl`).

## Example 1 — version + `CUTLASS_PATH` headers

In [ ]:
import os
import cutlass_cppgen
import cutlass_library

cutlass_path = cutlass_cppgen.CUTLASS_PATH
print("cutlass_cppgen.__version__ :", cutlass_cppgen.__version__)
print("CUTLASS_PATH               :", cutlass_path)

assert isinstance(cutlass_cppgen.__version__, str) and cutlass_cppgen.__version__[0].isdigit(), cutlass_cppgen.__version__
assert os.path.isdir(cutlass_path), f"CUTLASS_PATH is not a directory: {cutlass_path}"
assert os.path.isdir(os.path.join(cutlass_path, "include")), "missing include/ under CUTLASS_PATH"
print("OK — version + CUTLASS_PATH headers found")

## Example 2 — enumerate CUTLASS types (`cutlass_library` tag tables)

`cutlass_library` is the pure-Python code-generation layer (no CUDA context required). Its
numeric-type / layout / opcode tables are what name and emit every kernel. Enumerating them
exercises the real code-gen runtime surface GPU-lessly.

In [ ]:
import cutlass_library as cl

assert cl.DataTypeSize[cl.DataType.f16] == 16
assert cl.DataTypeSize[cl.DataType.f32] == 32
assert cl.DataTypeTag[cl.DataType.f16] == "cutlass::half_t"
assert cl.DataTypeTag[cl.DataType.bf16] == "cutlass::bfloat16_t"
assert cl.LayoutTag[cl.LayoutType.RowMajor] == "cutlass::layout::RowMajor"
assert cl.OpcodeClassNames[cl.OpcodeClass.TensorOp] == "tensorop"
assert cl.GemmKindNames[cl.GemmKind.Universal] == "gemm"

for dt in (cl.DataType.f16, cl.DataType.bf16, cl.DataType.tf32, cl.DataType.f32, cl.DataType.s8):
    print(f"{cl.DataTypeNames[dt]:>5}  {cl.DataTypeSize[dt]:>3} bits  ->  {cl.DataTypeTag[dt]}")
print("known SM archs:", sorted(cl.ArchitectureNames)[:6], "...")
print("OK — cutlass_library type/layout/opcode tables enumerated")

## Example 3 — build a TensorOp GEMM operation and emit its CUDA C++

The heart of the `cutlass_library` / `cutlass_cppgen` Python surface is code generation: describe a
tiled TensorOp GEMM, then emit the CUDA C++ template instantiation that `cutlass_cppgen` would
JIT-compile. Runs entirely on CPU.

In [ ]:
import cutlass_library as cl
from cutlass_library import gemm_operation as go

math_inst = cl.MathInstruction(
    [16, 8, 16], cl.DataType.f16, cl.DataType.f16, cl.DataType.f32,
    cl.OpcodeClass.TensorOp, cl.MathOperation.multiply_add)
tile = cl.TileDescription([128, 128, 32], 3, [2, 2, 1], math_inst,
                          min_compute=80, max_compute=90)
A = cl.TensorDescription(cl.DataType.f16, cl.LayoutType.RowMajor, 8)
B = cl.TensorDescription(cl.DataType.f16, cl.LayoutType.ColumnMajor, 8)
C = cl.TensorDescription(cl.DataType.f32, cl.LayoutType.RowMajor, 4)

op = go.GemmOperation(cl.GemmKind.Universal, tile.minimum_compute_capability,
                      tile, A, B, C, cl.DataType.f32)
name = op.procedural_name()
src = go.EmitGemmInstance().emit(op)
print("procedural_name:", name)
print("emitted C++ chars:", len(src))
assert name == "cutlass_tensorop_s16816gemm_f16_128x128_32x3_tn_align8", name
assert "cutlass::gemm::device::Gemm<" in src
assert "cutlass::half_t" in src and "cutlass::layout::RowMajor" in src
print("OK — GEMM operation emitted valid CUTLASS C++")

## Example 4 — build a Conv2d Fprop operation and emit its CUDA C++

Same code-gen path for the implicit-GEMM convolution family: describe an NHWC f16 TensorOp Conv2d
forward-propagation kernel and emit its CUTLASS C++ instantiation. CPU-only.

In [ ]:
import cutlass_library as cl
from cutlass_library import conv2d_operation as co

math_inst = cl.MathInstruction(
    [16, 8, 16], cl.DataType.f16, cl.DataType.f16, cl.DataType.f32,
    cl.OpcodeClass.TensorOp, cl.MathOperation.multiply_add)
tile = cl.TileDescription([128, 128, 32], 3, [2, 2, 1], math_inst,
                          min_compute=80, max_compute=90)
A = cl.TensorDescription(cl.DataType.f16, cl.LayoutType.TensorNHWC, 8)
B = cl.TensorDescription(cl.DataType.f16, cl.LayoutType.TensorNHWC, 8)
C = cl.TensorDescription(cl.DataType.f32, cl.LayoutType.TensorNHWC, 4)

conv = co.Conv2dOperation(
    cl.ConvKind.Fprop, cl.IteratorAlgorithm.Optimized, tile.minimum_compute_capability,
    tile, A, B, C, cl.DataType.f32, cl.StrideSupport.Strided)
name = conv.procedural_name()
src = co.EmitConv2dInstance().emit(conv)
print("conv procedural_name:", name)
print("emitted C++ chars:", len(src))
assert "fprop" in name and "nhwc" in name, name
assert "cutlass::conv" in src
print("OK — Conv2d Fprop operation emitted valid CUTLASS C++")